# BTC Volume Pattern Checker (dla Google Colab)

**Instrukcja:** Zmień wartość TIMEFRAME poniżej (np. "5m", "15m", "1h"), potem uruchom całą komórkę. Działa w Colab. Sprawdza warunek: zmiana koloru słupka volume + wyższy volume niż poprzedni bar tego samego koloru.

In [ ]:
# === USTAWIENIA (ZMIEŃ TUTAJ) ===
TIMEFRAME = "1m"          # możesz zmienić na: "5m", "15m", "1h", "6h", "1d"

import requests
from datetime import datetime

GRANULARITY_MAP = {
    "1m": 60,
    "5m": 300,
    "15m": 900,
    "1h": 3600,
    "6h": 21600,
    "1d": 86400
}
granularity = GRANULARITY_MAP.get(TIMEFRAME, 60)

def get_bars(timeframe):
    url = f"https://api.exchange.coinbase.com/products/BTC-USD/candles?granularity={granularity}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        if not isinstance(data, list) or len(data) == 0:
            print("Błąd API lub pusta odpowiedź:", data)
            return []
        data = data[-10:]  # ostatnie 10 barów
        bars = []
        for bar in data:
            open_time = datetime.fromtimestamp(bar[0]).strftime('%H:%M')
            o = float(bar[3])
            c = float(bar[4])
            v = float(bar[5])
            color = "green" if c > o else "red"
            bars.append({"time": open_time, "open": o, "close": c, "volume": v, "color": color})
        return bars
    except Exception as e:
        print("Błąd pobierania:", e)
        return []

def check_volume_pattern(bars):
    if len(bars) < 2:
        return "Za mało danych do analizy."
    for i in range(1, len(bars)):
        curr = bars[i]
        prev = bars[i-1]
        if curr["color"] != prev["color"]:
            last_same_color_vol = 0
            for j in range(i-1, -1, -1):
                if bars[j]["color"] == curr["color"]:
                    last_same_color_vol = bars[j]["volume"]
                    break
            if curr["volume"] > last_same_color_vol:
                return f"WZORZEC WYKRYTY! Bar {curr['time']}: kolor {curr['color']}, volume {curr['volume']:.2f} > poprzedni {last_same_color_vol:.2f}"
    return "Brak wzorca w ostatnich 10 barach."

# === URUCHOMIENIE ===
print(f"=== BTC Volume Pattern Checker - Timeframe: {TIMEFRAME} ===")
bars = get_bars(TIMEFRAME)
if bars:
    print("Ostatnie 10 barów (najnowszy na końcu):")
    for b in bars:
        print(f"{b['time']} | {b['color']:6} | vol: {b['volume']:.2f}")
    print()
    result = check_volume_pattern(bars)
    print(result)
else:
    print("Nie udało się pobrać danych.")